<a href="https://colab.research.google.com/github/abhi8823/machine-learning-lab/blob/main/Spam_Ham_filtering_model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

class SMSModerationPipeline:
    def __init__(self, data_url: str):
        self.data_url = data_url
        self.vectorizer = TfidfVectorizer(stop_words='english', sublinear_tf=True, ngram_range=(1, 2))
        self.model = LogisticRegression(class_weight='balanced', random_state=42)
        self.is_trained = False

    def load_data(self) -> pd.DataFrame:
        """Loads and prepares the SMS dataset from the source URL."""
        df = pd.read_csv(self.data_url, sep='\t', header=None, names=['label', 'message'])
        df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
        print(f"Successfully loaded and initialized {len(df)} rows from the dataset.")
        return df

    def train(self, df: pd.DataFrame):
        """Trains the TF-IDF vectorizer and Logistic Regression classifier."""
        X_train, X_test, y_train, y_test = train_test_split(
            df['message'], df['label_num'], test_size=0.20, random_state=42, stratify=df['label_num']
        )

        # Fit vectorizer and transform training features
        X_train_vec = self.vectorizer.fit_transform(X_train)
        X_test_vec = self.vectorizer.transform(X_test)

        # Train model
        self.model.fit(X_train_vec, y_train)
        self.is_trained = True

        # Evaluate performance
        predictions = self.model.predict(X_test_vec)
        acc = accuracy_score(y_test, predictions)

        print(f"\nPipeline Training Complete! Test Accuracy: {acc * 100:.2f}%")
        print("\nDetailed Evaluation Metrics:")
        print(classification_report(y_test, predictions, target_names=['Ham', 'Spam']))

    def predict(self, messages: list) -> list:
        """Predicts whether new text messages are Spam or Ham."""
        if not self.is_trained:
            raise ValueError("Model must be trained before running predictions.")

        vec_messages = self.vectorizer.transform(messages)
        preds = self.model.predict(vec_messages)
        return ["Spam" if p == 1 else "Ham" for p in preds]


if __name__ == "__main__":
    # Dataset source reference
    DATA_SOURCE = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

    # Initialize and run pipeline
    pipeline = SMSModerationPipeline(DATA_SOURCE)
    dataset = pipeline.load_data()
    pipeline.train(dataset)

    # Inference test
    sample_texts = [
        "Free entry in 2 a wkly comp to win FA Cup final tkts",
        "Hey, are we still meeting for lunch today?"
    ]

    results = pipeline.predict(sample_texts)
    print("Sample Predictions:")
    for text, label in zip(sample_texts, results):
        print(f"Message: '{text}' --> Prediction: {label}")

Successfully loaded and initialized 5572 rows from the dataset.

Pipeline Training Complete! Test Accuracy: 97.76%

Detailed Evaluation Metrics:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99       966
        Spam       0.90      0.93      0.92       149

    accuracy                           0.98      1115
   macro avg       0.95      0.96      0.95      1115
weighted avg       0.98      0.98      0.98      1115

Sample Predictions:
Message: 'Free entry in 2 a wkly comp to win FA Cup final tkts' --> Prediction: Spam
Message: 'Hey, are we still meeting for lunch today?' --> Prediction: Ham
